# MarsWalk · AI4Mars ile kum ve kaya tespiti (kask kamerası denemesi)

Curiosity'nin Navcam fotoğraflarında her pikseli **soil / bedrock / sand / big rock** olarak sınıflandıran bir
segmentasyon modeli eğitir, NASA'nın uzman etiketli test setinde ölçer ve MarsWalk'un üç bilim durağının
fotoğraflarına uygular.

- **Veri:** AI4Mars (NASA JPL, Swan ve diğ. 2021), Curiosity Navcam, gönüllü etiketli eğitim seti + 322 görüntülük
  uzman test seti. Hugging Face kopyası: `hassanjbara/AI4MARS` (CC0).
- **Model:** DeepLabv3 + MobileNetV3-Large (torchvision, COCO üzerinde önceden eğitilmiş), 4 sınıfa ince ayar.
- **Neden YOLO değil:** Kum bir nesne değil, bir alan; kutu yerine her piksel sınıflandırılıyor (anlamsal segmentasyon).

**Çalıştırma**
1. Colab'da: *Runtime → Change runtime type → T4 GPU*.
2. *Runtime → Run all*. Toplam tahminen 30–60 dk (indirme ~5 dk, hazırlık ~5–10 dk, eğitim). İlk turun süresi
   ekrana yazılır; bir tur 5 dakikadan uzun sürerse `EPOCHS` ya da `N_SHARDS` değerini düşür.
3. Sonunda `marswalk_ai4mars_outputs.zip` iner: durak fotoğraflarının boyalı halleri, test sonuçları, model.

Bağlantı koparsa: her turdan sonra en iyi model `/content/ai4mars_deeplab.pt` dosyasına kaydedilir; `RESUME = True`
yapıp eğitim hücresinden devam edebilirsin.

In [ ]:
# Settings. Lower N_SHARDS or EPOCHS if time is short (each shard ~1,030 labelled images, ~475 MB).
# The epoch time is printed after the first epoch.
import os
SMOKE = os.environ.get("SMOKE") == "1"   # tiny CPU run used to test this notebook; leave off in Colab

N_SHARDS = 4        # training shards to download (of 13)
IMG = 512           # images and masks are resized to IMG x IMG
EPOCHS = 10
BATCH = 16
LR = 4e-4
RESUME = False      # True: load /content/ai4mars_deeplab.pt and continue
WORK = "/content" if os.path.isdir("/content") else os.path.abspath("work")

if SMOKE:
    N_SHARDS, IMG, EPOCHS, BATCH = 1, 128, 1, 4
os.makedirs(WORK, exist_ok=True)
CLASSES = ["soil", "bedrock", "sand", "big rock"]
IGNORE = 255
HF = "https://huggingface.co/api/datasets/hassanjbara/AI4MARS/parquet/default"
REPO_RAW = "https://raw.githubusercontent.com/SaraT-7/NASA_Hackathon_2026/main"
print("work dir:", WORK, "| smoke test" if SMOKE else "")

In [ ]:
import io, json, math, time, urllib.request
import numpy as np
import pyarrow.parquet as pq
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
import torchvision
from torchvision.models.segmentation import deeplabv3_mobilenet_v3_large, DeepLabV3_MobileNet_V3_Large_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| device:", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))
elif not SMOKE:
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then run again.")

## 1 · Veriyi indir

In [ ]:
def download(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    t = time.time()
    req = urllib.request.Request(url, headers={"User-Agent": "marswalk-notebook"})
    with urllib.request.urlopen(req) as r, open(path + ".part", "wb") as f:
        while True:
            chunk = r.read(1 << 22)
            if not chunk:
                break
            f.write(chunk)
    os.replace(path + ".part", path)
    print(f"  {os.path.basename(path)}: {os.path.getsize(path) / 1e6:.0f} MB in {time.time() - t:.0f} s")
    return path

train_files = [download(f"{HF}/train/{i}.parquet", f"{WORK}/train{i}.parquet") for i in range(N_SHARDS)]
test_files = {s: download(f"{HF}/{s}/0.parquet", f"{WORK}/{s}.parquet") for s in ["test_min1", "test_min2", "test_min3"]}

## 2 · Görüntüleri hazırla

Etiketsiz pikseller, rover'ın kendi gövdesi (`rover_mask`) ve 30 m'den uzak kısımlar (`range_mask`) kayba girmez.

In [ ]:
def decode(cell, mode):
    return Image.open(io.BytesIO(cell["bytes"])).convert(mode) if cell is not None else None

def load_split(path, limit=None):
    imgs, labs = [], []
    pf = pq.ParquetFile(path)
    for batch in pf.iter_batches(batch_size=64):
        for r in batch.to_pylist():
            if not r["has_labels"]:
                continue
            im = decode(r["image"], "L").resize((IMG, IMG), Image.BILINEAR)
            lab = np.array(decode(r["label_mask"], "L").resize((IMG, IMG), Image.NEAREST))
            for key in ("rover_mask", "range_mask"):
                m = decode(r[key], "L")
                if m is not None:
                    lab[np.array(m.resize((IMG, IMG), Image.NEAREST)) > 0] = IGNORE
            lab[(lab > 3) & (lab != IGNORE)] = IGNORE
            imgs.append(np.array(im, dtype=np.uint8)); labs.append(lab)
            if limit and len(imgs) >= limit:
                return np.stack(imgs), np.stack(labs)
    return np.stack(imgs), np.stack(labs)

t = time.time()
lim = 24 if SMOKE else None
parts = [load_split(p, lim) for p in train_files]
X_train = np.concatenate([p[0] for p in parts]); Y_train = np.concatenate([p[1] for p in parts]); del parts
tests = {s: load_split(p, 8 if SMOKE else None) for s, p in test_files.items()}
print(f"train: {len(X_train)} images | " + " | ".join(f"{s}: {len(v[0])}" for s, v in tests.items())
      + f" | {time.time() - t:.0f} s")

freq = np.bincount(Y_train[Y_train != IGNORE].ravel(), minlength=4).astype(np.float64)
share = freq / freq.sum()
print("labelled pixel share:", {c: f"{s:.1%}" for c, s in zip(CLASSES, share)})
# Rare classes (sand, big rock) get more weight in the loss; clipped so big rock does not dominate.
class_w = np.clip(np.median(share) / np.maximum(share, 1e-6), 0.5, 5.0)
print("class weights:", {c: round(float(w), 2) for c, w in zip(CLASSES, class_w)})

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def to_input(x_uint8):
    # Navcam is greyscale: copy it to 3 channels for the ImageNet/COCO-pretrained backbone.
    x = torch.as_tensor(x_uint8, dtype=torch.float32).div(255).unsqueeze(1).repeat(1, 3, 1, 1)
    return (x - MEAN.to(x.device)) / STD.to(x.device)

class Navcam(torch.utils.data.Dataset):
    def __init__(self, X, Y, augment):
        self.X, self.Y, self.augment = X, Y, augment
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        x, y = self.X[i], self.Y[i]
        if self.augment:
            if np.random.rand() < 0.5:
                x, y = x[:, ::-1], y[:, ::-1]
            gain, bias = np.random.uniform(0.8, 1.2), np.random.uniform(-20, 20)
            x = np.clip(x.astype(np.float32) * gain + bias, 0, 255)
        return np.ascontiguousarray(x, dtype=np.uint8), np.ascontiguousarray(y, dtype=np.int64)

workers = 0 if SMOKE else 2
train_dl = torch.utils.data.DataLoader(Navcam(X_train, Y_train, True), batch_size=BATCH, shuffle=True,
                                       num_workers=workers, drop_last=True, pin_memory=device == "cuda")

## 3 · Model ve eğitim

In [ ]:
model = deeplabv3_mobilenet_v3_large(weights=DeepLabV3_MobileNet_V3_Large_Weights.DEFAULT)
model.classifier[4] = nn.Conv2d(model.classifier[4].in_channels, 4, 1)
model.aux_classifier[4] = nn.Conv2d(model.aux_classifier[4].in_channels, 4, 1)
model = model.to(device)
CKPT = f"{WORK}/ai4mars_deeplab.pt"
if RESUME and os.path.exists(CKPT):
    model.load_state_dict(torch.load(CKPT, map_location=device)["model"])
    print("resumed from", CKPT)

loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_w, dtype=torch.float32, device=device), ignore_index=IGNORE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
steps = EPOCHS * len(train_dl)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.1)
scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")

@torch.no_grad()
def predict(X, bs=16):
    model.eval()
    out = []
    for i in range(0, len(X), bs):
        x = to_input(X[i:i + bs]).to(device)
        with torch.autocast(device_type="cuda", enabled=device == "cuda"):
            out.append(model(x)["out"].argmax(1).cpu().numpy().astype(np.uint8))
    return np.concatenate(out)

def scores(pred, Y):
    ok = Y != IGNORE
    cm = np.bincount(Y[ok].astype(np.int64) * 4 + pred[ok], minlength=16).reshape(4, 4)
    iou = np.diag(cm) / np.maximum(cm.sum(0) + cm.sum(1) - np.diag(cm), 1)
    present = cm.sum(1) > 0
    return {"pixel_acc": float(np.diag(cm).sum() / cm.sum()), "mIoU": float(iou[present].mean()),
            **{f"IoU {c}": float(v) for c, v in zip(CLASSES, iou)}, "confusion": cm.tolist()}

In [ ]:
best, history = -1.0, []
for ep in range(EPOCHS):
    model.train(); t = time.time(); total = 0.0
    for x, y in train_dl:
        x, y = to_input(x.numpy()).to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", enabled=device == "cuda"):
            out = model(x)
            loss = loss_fn(out["out"], y) + 0.4 * loss_fn(out["aux"], y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        total += loss.item()
    s = scores(predict(tests["test_min3"][0]), tests["test_min3"][1])
    history.append({"epoch": ep + 1, "loss": total / len(train_dl), **{k: v for k, v in s.items() if k != "confusion"}})
    print(f"epoch {ep + 1}/{EPOCHS}  loss {total / len(train_dl):.3f}  test_min3 mIoU {s['mIoU']:.3f}  "
          f"sand IoU {s['IoU sand']:.3f}  pixel acc {s['pixel_acc']:.3f}  ({time.time() - t:.0f} s)")
    if s["mIoU"] > best:
        best = s["mIoU"]
        torch.save({"model": model.state_dict(), "img": IMG, "classes": CLASSES, "epoch": ep + 1}, CKPT)
model.load_state_dict(torch.load(CKPT, map_location=device)["model"])
print("best test_min3 mIoU:", round(best, 3))

## 4 · NASA'nın uzman test setinde sonuç

Üç test bölümü, uzman etiketlerinin farklı uzlaşı seviyeleridir. Sunumda dürüstçe bu sayıları veriyoruz.

In [ ]:
results = {s: scores(predict(X), Y) for s, (X, Y) in tests.items()}
print(f"{'split':10s} {'pixel acc':>9s} {'mIoU':>6s} " + " ".join(f"{c:>9s}" for c in CLASSES))
for s, r in results.items():
    print(f"{s:10s} {r['pixel_acc']:9.3f} {r['mIoU']:6.3f} " + " ".join(f"{r['IoU ' + c]:9.3f}" for c in CLASSES))
print("\nconfusion (test_min3; rows = expert label, columns = model):")
for c, row in zip(CLASSES, results["test_min3"]["confusion"]):
    print(f"  {c:9s}", row)

In [ ]:
import matplotlib; matplotlib.use("Agg") if SMOKE else None
import matplotlib.pyplot as plt
# soil: no colour; bedrock: blue; sand: yellow; big rock: red
PALETTE = np.array([[0, 0, 0, 0], [40, 120, 255, 110], [255, 210, 0, 150], [255, 40, 40, 190]], dtype=np.uint8)

def overlay(gray, pred, valid=None):
    base = np.stack([gray] * 3, -1).astype(np.float32)
    col = PALETTE[pred].astype(np.float32)
    a = col[..., 3:] / 255
    if valid is not None:
        a = a * valid[..., None]
    return (base * (1 - a) + col[..., :3] * a).astype(np.uint8)

X3, Y3 = tests["test_min3"]
idx = np.linspace(0, len(X3) - 1, 4).astype(int)
P3 = predict(X3[idx])
fig, ax = plt.subplots(len(idx), 3, figsize=(10, 3.3 * len(idx)))
for k, i in enumerate(idx):
    ax[k, 0].imshow(X3[i], cmap="gray"); ax[k, 0].set_title("Navcam")
    lab = np.where(Y3[i] == IGNORE, 0, Y3[i])
    ax[k, 1].imshow(overlay(X3[i], lab, (Y3[i] != IGNORE).astype(np.float32))); ax[k, 1].set_title("expert label")
    ax[k, 2].imshow(overlay(X3[i], P3[k])); ax[k, 2].set_title("model")
    for a in ax[k]: a.axis("off")
plt.tight_layout(); plt.savefig(f"{WORK}/test_examples.png", dpi=90); plt.show()

## 5 · MarsWalk duraklarına uygula

Durak fotoğrafları, Curiosity'nin bu duraklarda çektiği gerçek Navcam görüntüleri (`app_data/stops.json`). Oranlar görüntünün alt yarısından (yaklaşık 30 m içi, eğitim verisiyle aynı menzil) hesaplanır; gökyüzü ve ufuk dahil edilmez.

In [ ]:
def fetch_image(urls):
    for u in urls:
        try:
            req = urllib.request.Request(u, headers={"User-Agent": "marswalk-notebook"})
            with urllib.request.urlopen(req, timeout=60) as r:
                return Image.open(io.BytesIO(r.read())).convert("L"), u
        except Exception as e:
            print("  could not load", u, "-", e)
    return None, None

stops = json.loads(urllib.request.urlopen(f"{REPO_RAW}/app_data/stops.json", timeout=60).read())["stops"]
OUT = f"{WORK}/marswalk_ai4mars"; os.makedirs(OUT, exist_ok=True)
stop_results = []
for s in stops:
    ph = s.get("photo") or {}
    urls = [u for u in [ph.get("source_url"), f"{REPO_RAW}/app_data/{ph['file']}" if ph.get("file") else None] if u]
    im, used = fetch_image(urls)
    if im is None:
        continue
    small = np.array(im.resize((IMG, IMG), Image.BILINEAR))
    pred_small = predict(small[None])[0]
    pred = np.array(Image.fromarray(pred_small).resize(im.size, Image.NEAREST))
    gray = np.array(im)
    near = pred[pred.shape[0] // 2:]
    pct = {c: round(float((near == k).mean() * 100), 1) for k, c in enumerate(CLASSES)}
    # Colour only the near field: the model never saw sky or ground beyond 30 m, so it would paint them at random.
    valid = np.zeros(pred.shape, dtype=np.float32); valid[pred.shape[0] // 2:] = 1
    painted = overlay(gray, pred, valid)
    painted[pred.shape[0] // 2 - 1:pred.shape[0] // 2 + 1] = (255, 255, 255)
    Image.fromarray(painted).save(f"{OUT}/{s['id']}_terrain.png")
    side = np.concatenate([np.stack([gray] * 3, -1), painted], axis=1)
    Image.fromarray(side).save(f"{OUT}/{s['id']}_side_by_side.jpg", quality=88)
    stop_results.append({"id": s["id"], "name": s["name"], "sol": ph.get("sol"), "image_url": used,
                         "near_field_percent": pct})
    print(f"{s['id']} {s['name']} (sol {ph.get('sol')}): near field " + ", ".join(f"{c} {v}%" for c, v in pct.items()))

fig, ax = plt.subplots(len(stop_results), 1, figsize=(12, 6 * len(stop_results)))
ax = np.atleast_1d(ax)
for a, r in zip(ax, stop_results):
    a.imshow(Image.open(f"{OUT}/{r['id']}_side_by_side.jpg")); a.axis("off")
    a.set_title(f"{r['name']} · sand {r['near_field_percent']['sand']}% · big rock {r['near_field_percent']['big rock']}%"
                "  (yellow = sand, red = big rock, blue = bedrock; lower half only)")
plt.tight_layout(); plt.show()

## 6 · Sonuçları indir

In [ ]:
import shutil
summary = {
    "model": "DeepLabv3 + MobileNetV3-Large (torchvision, COCO-pretrained), fine-tuned on AI4Mars",
    "dataset": "AI4Mars (NASA JPL; Swan et al. 2021, doi:10.1109/CVPRW53098.2021.00226), HF hassanjbara/AI4MARS, CC0",
    "train_images": int(len(X_train)), "img_size": IMG, "epochs": EPOCHS, "class_weights": class_w.round(3).tolist(),
    "test": {s: {k: v for k, v in r.items() if k != "confusion"} for s, r in results.items()},
    "confusion_test_min3": results["test_min3"]["confusion"],
    "history": history, "stops": stop_results,
    "note": "Near-field percentages use the lower half of each photo. Experimental; not used in the route cost.",
}
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=2)
shutil.copy(f"{WORK}/test_examples.png", OUT)
shutil.copy(CKPT, OUT)
zip_path = shutil.make_archive(f"{WORK}/marswalk_ai4mars_outputs", "zip", OUT)
print(zip_path, f"{os.path.getsize(zip_path) / 1e6:.0f} MB")
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass